In [3]:
"""Experiements:
1. comparison with stella on gpt3.5
-- Recommender AI Agent: Integrating Large Language Models for Interactive Recommendations
2. Ablation study: calculating bias on one dataset, and using it to rerank the other
2b. ablation study: reranking without the bias calculation
XXX 3. adding user and movie metadata into the prompt XXX
-- study of biases of different LLMs
4. comparison with Frank et al
5. Comparison with Permutation Self-Consistency
6. comparison with LLM as judge results
7.comparison with Sun et al.
8. Comparison with Large Language Models are Zero-Shot Rankers for Recommender Systems
9. bias detection, number of unique users used for calculation
10. : Ablation Study about Length of Ensemble Steps for Probing Detection Set.
11. Accuracy vs number of trails.
12. bias before and after correction.
"""


'Experiements:\n1. comparison with stella on gpt3.5\n-- Recommender AI Agent: Integrating Large Language Models for Interactive Recommendations\n2. Ablation study: calculating bias on one dataset, and using it to rerank the other\n2b. ablation study: reranking without the bias calculation\n3. adding user and movie metadata into the prompt\n-- study of biases of different LLMs\n4. comparison with Frank et al\n5. Comparison with Permutation Self-Consistency\n6. comparison with LLM as judge results\n7.comparison with Sun et al.\n8. Comparison with Large Language Models are Zero-Shot Rankers for Recommender Systems\n9. bias detection, number of unique users used for calculation\n10. : Ablation Study about Length of Ensemble Steps for Probing Detection Set.\n11. Accuracy vs number of trails.\n'

In [4]:
import pandas as pd

# Define file paths
base_path = 'data/ml-1m/'
ratings_file = base_path + 'ratings.dat'
users_file = base_path + 'users.dat'
movies_file = base_path + 'movies.dat'

ratings = pd.read_csv(
    ratings_file,
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    encoding='latin-1'
)

users = pd.read_csv(
    users_file,
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
    encoding='latin-1'
)

movies = pd.read_csv(
    movies_file,
    sep='::',
    engine='python',
    names=['MovieID', 'Title', 'Genres'],
    encoding='latin-1'
)

# Show sample entries
print("Ratings:\n", ratings.head(), '\n')
print("Users:\n", users.head(), '\n')
print("Movies:\n", movies.head())

Ratings:
    UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291 

Users:
    UserID Gender  Age  Occupation Zip-code
0       1      F    1          10    48067
1       2      M   56          16    70072
2       3      M   25          15    55117
3       4      M   45           7    02460
4       5      M   25          20    55455 

Movies:
    MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [5]:
# Merge ratings with users
ratings_users = pd.merge(ratings, users, on='UserID')

# Merge with movies
full_data = pd.merge(ratings_users, movies, on='MovieID')

print("\nCombined Dataset:\n", full_data[full_data['UserID'] == 1][['Title', 'Timestamp']])


Combined Dataset:
                                                 Title  Timestamp
0              One Flew Over the Cuckoo's Nest (1975)  978300760
1                    James and the Giant Peach (1996)  978302109
2                                 My Fair Lady (1964)  978301968
3                              Erin Brockovich (2000)  978300275
4                                Bug's Life, A (1998)  978824291
5                          Princess Bride, The (1987)  978302268
6                                      Ben-Hur (1959)  978302039
7                           Christmas Story, A (1983)  978300719
8              Snow White and the Seven Dwarfs (1937)  978302268
9                            Wizard of Oz, The (1939)  978301368
10                        Beauty and the Beast (1991)  978824268
11                                        Gigi (1958)  978301752
12                      Miracle on 34th Street (1947)  978302281
13                    Ferris Bueller's Day Off (1986)  978302124
14   

In [6]:
from LLM_debias import build_prompt, LLMPositionBiasAnalyzer

In [7]:
analyzer = LLMPositionBiasAnalyzer(data=full_data,data_name='movie_lens', num_shuffles=50, list_size=100, model='gpt-3.5-turbo',backend='openai')

Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: basic
  Rate Limits: 500 RPM, 200000 TPM
  Max Workers: 5
  Batch Size: 10
  Request Delay: 0.150s


In [8]:
candidate_list,user_items, last_item = analyzer.create_candidate_list(1)
print(user_items[-5:])
print(candidate_list)
print(last_item)

[{'title': 'Random Hearts (1999)', 'original_position': 0}, {'title': "They Shoot Horses, Don't They? (1969)", 'original_position': 1}, {'title': 'Wilde (1997)', 'original_position': 2}, {'title': 'Single Girl, A (La Fille Seule) (1995)', 'original_position': 3}, {'title': 'Down to You (2000)', 'original_position': 4}, {'title': 'Mission to Mars (2000)', 'original_position': 5}, {'title': 'Dave (1993)', 'original_position': 6}, {'title': 'Contempt (Le Mépris) (1963)', 'original_position': 7}, {'title': 'Mommie Dearest (1981)', 'original_position': 8}, {'title': 'Clean Slate (Coup de Torchon) (1981)', 'original_position': 9}, {'title': 'Last Man Standing (1996)', 'original_position': 10}, {'title': 'Trial by Jury (1994)', 'original_position': 11}, {'title': 'Christine (1983)', 'original_position': 12}, {'title': 'Green Mile, The (1999)', 'original_position': 13}, {'title': 'Race the Sun (1996)', 'original_position': 14}, {'title': 'Wonderland (1997)', 'original_position': 15}, {'title':

In [20]:
prompt = build_prompt('movie_lens',user_items[-5:],candidate_list)
print(prompt)

You are a movie recommendation system. Rerank all the candidates from most to least recommended.
Return JSON {"ranked_movies": [10, 5, 2, ...]} with movie numbers in order of preference.

User viewing history: Antz (1998), Hunchback of Notre Dame, The (1996), Bug's Life, A (1998), Mulan (1998), Hercules (1997)

Movies to rank:
1. Random Hearts (1999)
2. They Shoot Horses, Don't They? (1969)
3. Wilde (1997)
4. Single Girl, A (La Fille Seule) (1995)
5. Down to You (2000)
6. Mission to Mars (2000)
7. Dave (1993)
8. Contempt (Le Mépris) (1963)
9. Mommie Dearest (1981)
10. Clean Slate (Coup de Torchon) (1981)
11. Last Man Standing (1996)
12. Trial by Jury (1994)
13. Christine (1983)
14. Green Mile, The (1999)
15. Race the Sun (1996)
16. Wonderland (1997)
17. Believers, The (1987)
18. Crash (1996)
19. Great White Hype, The (1996)
20. Reluctant Debutante, The (1958)
21. GoldenEye (1995)
22. Bewegte Mann, Der (1994)
23. Open Season (1996)
24. Convent, The (Convento, O) (1995)
25. Steel Magnoli

In [8]:
rank_order, reranked_list = analyzer.llm_reranking(prompt, candidate_list)

In [9]:
print(rank_order)
print(reranked_list)

[14, 6, 11, 29, 44, 19, 32, 35, 72, 93, 80, 61, 7, 75, 65, 39, 40, 36, 58, 79, 87, 64, 15, 12, 22, 16, 17, 92, 37, 46, 47, 41, 55, 85, 70, 42, 30, 89, 88, 78, 56, 74, 10, 38, 50, 51, 52, 53, 54, 57, 59, 60, 62, 63, 66, 67, 68, 69, 71, 73, 76, 77, 81, 82, 83, 84, 86, 90, 91, 94, 95, 96, 97, 98, 99, 100, 1, 2, 3, 4, 5, 8, 9, 13, 18, 20, 21, 23, 24, 25, 26, 27, 28, 31, 33, 34, 43, 45, 48, 49]
[{'title': 'Green Mile, The (1999)', 'original_position': 13, 'llm_score': 1.0, 'llm_rank': 1}, {'title': 'Mission to Mars (2000)', 'original_position': 5, 'llm_score': 0.98989898989899, 'llm_rank': 2}, {'title': 'Last Man Standing (1996)', 'original_position': 10, 'llm_score': 0.9797979797979798, 'llm_rank': 3}, {'title': 'Mighty Joe Young (1998)', 'original_position': 28, 'llm_score': 0.9696969696969697, 'llm_rank': 4}, {'title': 'Rounders (1998)', 'original_position': 43, 'llm_score': 0.9595959595959596, 'llm_rank': 5}, {'title': 'Great White Hype, The (1996)', 'original_position': 18, 'llm_score'

In [ ]:
experiment_result = analyzer.run_bias_detection_experiment(1)


Running bias detection experiment with 50 shuffles...


Processing shuffles:   4%|▉                      | 2/50 [00:08<03:46,  4.71s/it]

In [11]:
prebias_gpt35_movielens = {'avg_primacy': 6.063,
 'avg_recency': 3.336,
 'avg_middle': 0.628}

In [13]:
# FULL-FLEDGED EVALUATION - Compare with Previous Studies
# Comprehensive evaluation matching research standards

print("🚀 COMPREHENSIVE EVALUATION: OUR DEBIASED RANKING vs PREVIOUS STUDIES")
print("=" * 70)

print("📋 EVALUATION CONFIGURATION:")
print("  • Bias detection users: 50 (robust bias estimation)")
print("  • Evaluation users: 200 (statistically significant)")
print("  • Candidates per evaluation: 20 (standard)")
print("  • Randomization trials: 20 (matching paper methodology)")
print("  • Aggregation method: mean")
print("  • Separate user sets: No data leakage")
print("\n⏳ Starting comprehensive evaluation...")

# Run full evaluation with research-grade parameters
results = analyzer.evaluate_our_method(
    num_bias_users=5,       # Robust bias detection
    num_eval_users=200,      # Statistically significant
    num_candidates=20,       # Standard candidate set
    num_trials=20,           # Matching paper methodology
    aggregation_method="mean",
    precalculated_bias = prebias_gpt35_movielens
)

# Extract key results
bias_analysis = results['bias_analysis']
our_method = results['our_method_evaluation']
config = results['method_config']

print(f"\n📊 BIAS DETECTION RESULTS:")
print(f"  Primacy bias: {bias_analysis['bias_scores']['primacy_bias']:.4f}")
print(f"  Recency bias: {bias_analysis['bias_scores']['recency_bias']:.4f}")
print(f"  Middle-ignoring bias: {bias_analysis['bias_scores']['middle_ignoring_bias']:.4f}")
print(f"  Users for bias detection: {bias_analysis['num_bias_users']}")

print(f"\n🎯 OUR METHOD PERFORMANCE (Movie Dataset):")
print(f"  Accuracy:    {our_method['accuracy']['mean']:.4f} ± {our_method['accuracy']['std']:.4f}")
print(f"  NDCG@1:      {our_method['ndcg_1']['mean']:.4f} ± {our_method['ndcg_1']['std']:.4f}")
print(f"  NDCG@5:      {our_method['ndcg_5']['mean']:.4f} ± {our_method['ndcg_5']['std']:.4f}")
print(f"  NDCG@10:     {our_method['ndcg_10']['mean']:.4f} ± {our_method['ndcg_10']['std']:.4f}")
print(f"  NDCG@20:     {our_method['ndcg_20']['mean']:.4f} ± {our_method['ndcg_20']['std']:.4f}")
print(f"  Evaluations: {our_method['accuracy']['num_evaluations']} users")

print(f"\n🏆 COMPARISON WITH PREVIOUS STUDIES (Movie Dataset):")
print("-" * 60)
print(f"{'Method':<25} {'Accuracy':<12} {'Difference':<12} {'Status'}")
print("-" * 60)

# Previous study benchmarks
benchmarks = {
    'Raw Output': {'acc': 0.2740, 'std': 0.0593},
    'Bootstrapping': {'acc': 0.2537, 'std': None},
    'STELLA': {'acc': 0.2976, 'std': None},
    'Our Method': {'acc': our_method['accuracy']['mean'], 'std': our_method['accuracy']['std']}
}

our_accuracy = our_method['accuracy']['mean']

for method, data in benchmarks.items():
    if method == 'Our Method':
        print(f"{method:<25} {data['acc']:.4f}±{data['std']:.4f} {'—':<12} {'🎯 OURS'}")
    else:
        diff = our_accuracy - data['acc']
        status = "✅ Better" if diff > 0 else "❌ Worse"
        std_str = f"±{data['std']:.4f}" if data['std'] else ""
        print(f"{method:<25} {data['acc']:.4f}{std_str:<7} {diff:+.4f}      {status}")

print("\n📈 NDCG PERFORMANCE ANALYSIS:")
ndcg_1 = our_method['ndcg_1']['mean']
if ndcg_1 > 0:
    print(f"  NDCG@5  retention:  {(our_method['ndcg_5']['mean']/ndcg_1*100):.1f}% of NDCG@1")
    print(f"  NDCG@10 retention:  {(our_method['ndcg_10']['mean']/ndcg_1*100):.1f}% of NDCG@1")
    print(f"  NDCG@20 retention:  {(our_method['ndcg_20']['mean']/ndcg_1*100):.1f}% of NDCG@1")

print(f"\n🎯 OVERALL RESEARCH ASSESSMENT:")
if our_accuracy > 0.2976:  # STELLA benchmark
    print("🌟 BREAKTHROUGH: Outperforms state-of-the-art STELLA!")
    improvement = ((our_accuracy - 0.2976) / 0.2976) * 100
    print(f"   📊 Improvement over STELLA: +{improvement:.1f}%")
elif our_accuracy > 0.2740:  # Raw Output benchmark  
    print("✅ STRONG: Significantly outperforms Raw Output baseline")
    improvement = ((our_accuracy - 0.2740) / 0.2740) * 100
    print(f"   📊 Improvement over Raw Output: +{improvement:.1f}%")
elif our_accuracy > 0.2537:  # Bootstrapping benchmark
    print("⚠️  MODERATE: Outperforms Bootstrapping only")
else:
    print("❌ NEEDS IMPROVEMENT: Below all previous studies")

print(f"\n✅ Comprehensive evaluation completed:")
print(f"   • Evaluated on {our_method['accuracy']['num_evaluations']} users")
print(f"   • {config['num_trials']} randomization trials per user") 
print(f"   • Debiased ranking based on input position bias")
print(f"   • Ready for publication comparison!")


🚀 COMPREHENSIVE EVALUATION: OUR DEBIASED RANKING vs PREVIOUS STUDIES
📋 EVALUATION CONFIGURATION:
  • Bias detection users: 50 (robust bias estimation)
  • Evaluation users: 200 (statistically significant)
  • Candidates per evaluation: 20 (standard)
  • Randomization trials: 20 (matching paper methodology)
  • Aggregation method: mean
  • Separate user sets: No data leakage

⏳ Starting comprehensive evaluation...
Starting evaluation with 200 users, 20 candidates, 20 trials
Parallel processing: True
  Max workers - Bias: 15, Trials: 10, Users: 3
Selected 5 bias users and 200 evaluation users
Using precalculated bias scores...
\nStep 2: Evaluating our method on 200 users...
Evaluating 200 users in parallel with max_workers=3...


User evaluation:   0%|                                  | 0/200 [00:00<?, ?it/s]


Running 20 randomization trials...
Executing 20 trials in parallel with max_workers=10...



als:   0%|                                            | 0/20 [00:00<?, ?it/s]


Running 20 randomization trials...
Executing 20 trials in parallel with max_workers=10...


...



Trials: 100%|███████████████████████████████████| 20/20 [00:05<00:00,  3.84it/s]


:  85%|█████████████████████████████▊     | 17/20 [00:03<00:00,  5.09it/s]

Completed 20 successful trials out of 20 attempted
Debiasing applied within each trial based on input position. Creating final ranking from aggregated debiased scores...
Final ranking created. Top 3 items:
  1. Rob Roy (1995) (agg: 0.680, debiased: 0.512, avg_weight: 0.813)
  2. Horse Whisperer, The (1998) (agg: 0.677, debiased: 0.502, avg_weight: 0.781)
  3. River Runs Through It, A (1992) (agg: 0.660, debiased: 0.499, avg_weight: 0.791)




Trials: 100%|███████████████████████████████████| 20/20 [00:04<00:00,  4.71it/s]


Completed 20 successful trials out of 20 attempted
Debiasing applied within each trial based on input position. Creating final ranking from aggregated debiased scores...
Final ranking created. Top 3 items:
  1. Rob Roy (1995) (agg: 0.620, debiased: 0.489, avg_weight: 0.833)
  2. Stripes (1981) (agg: 0.627, debiased: 0.485, avg_weight: 0.811)
  3. Rambo: First Blood Part II (1985) (agg: 0.630, debiased: 0.462, avg_weight: 0.772)


KeyboardInterrupt: 

In [1]:
# 🔄 BATCHED EVALUATION SYSTEM
# Process 200 users safely with checkpoint saving

print("🔄 BATCHED EVALUATION SYSTEM")
print("=" * 50)

print("✅ BENEFITS OF BATCHED PROCESSING:")
print("• Process users in small batches (e.g., 25 at a time)")
print("• Save progress after each batch")
print("• Resume from where you left off if interrupted")
print("• No risk of losing hours of work")
print("• Monitor progress in real-time")

print("\n📊 EXAMPLE USAGE:")
print("# Check if there's existing progress")
print("analyzer.get_checkpoint_status()")
print()
print("# Start batched evaluation")
print("results = analyzer.evaluate_our_method_batched(")
print("    num_eval_users=200,          # Total users to evaluate")
print("    batch_size=25,               # Users per batch")
print("    num_trials=20,               # Trials per user")
print("    precalculated_bias=prebias_gpt35_movielens,")
print("    checkpoint_file='my_evaluation.json'")
print(")")

print("\n⚡ BATCH PROCESSING FLOW:")
print("1. Batch 1: Users 1-25    → Save progress")
print("2. Batch 2: Users 26-50   → Save progress") 
print("3. Batch 3: Users 51-75   → Save progress")
print("4. ... (if interrupted, resume from last saved batch)")
print("5. Batch 8: Users 176-200 → Complete!")

print("\n💾 CHECKPOINT FEATURES:")
print("• Automatic saving after each batch")
print("• Resume from exact stopping point")
print("• Progress tracking and status reports")
print("• Bias analysis saved (computed only once)")
print("• Robust error handling")

print("\n🎯 READY TO RUN SAFELY!")
print("No more fear of losing progress on long evaluations!")


🔄 BATCHED EVALUATION SYSTEM
✅ BENEFITS OF BATCHED PROCESSING:
• Process users in small batches (e.g., 25 at a time)
• Save progress after each batch
• Resume from where you left off if interrupted
• No risk of losing hours of work
• Monitor progress in real-time

📊 EXAMPLE USAGE:
# Check if there's existing progress
analyzer.get_checkpoint_status()

# Start batched evaluation
results = analyzer.evaluate_our_method_batched(
    num_eval_users=200,          # Total users to evaluate
    batch_size=25,               # Users per batch
    num_trials=20,               # Trials per user
    precalculated_bias=prebias_gpt35_movielens,
    checkpoint_file='my_evaluation.json'
)

⚡ BATCH PROCESSING FLOW:
1. Batch 1: Users 1-25    → Save progress
2. Batch 2: Users 26-50   → Save progress
3. Batch 3: Users 51-75   → Save progress
4. ... (if interrupted, resume from last saved batch)
5. Batch 8: Users 176-200 → Complete!

💾 CHECKPOINT FEATURES:
• Automatic saving after each batch
• Resume from exa

In [9]:
# 📊 CHECK EXISTING PROGRESS
# Run this first to see if you have any saved progress

print("📊 Checking for existing evaluation progress...")

# Check if there's an existing checkpoint
status = analyzer.get_checkpoint_status("evaluation_200_users.json")

if status.get('status') == 'No checkpoint found':
    print("\n✨ Starting fresh - no previous progress found")
    print("Ready to begin batched evaluation!")
else:
    print(f"\n🔄 Found existing progress!")
    print(f"You can resume from where you left off.")
    
print("\n" + "="*50)


📊 Checking for existing evaluation progress...

✨ Starting fresh - no previous progress found
Ready to begin batched evaluation!



In [12]:
# 🚀 START BATCHED EVALUATION
# This will process 200 users in batches of 25, saving progress after each batch
# ⚠️ UNCOMMENT TO RUN ⚠️

print("🚀 STARTING BATCHED EVALUATION")
print("This will evaluate 200 users in batches of 25")
print("Progress will be saved after each batch!")
print()

# Uncomment the following lines to start the evaluation:

results_batched = analyzer.evaluate_our_method_batched(
    num_bias_users=5,                    # Quick bias detection
    num_eval_users=200,                  # Total users to evaluate 
    batch_size=25,                       # Users per batch (safe size)
    num_candidates=20,                   # Standard candidate set
    num_trials=20,                       # Full trials per user
    aggregation_method="mean",
    precalculated_bias=prebias_gpt35_movielens,  # Skip bias detection
    checkpoint_file="evaluation_200_users.json", # Save progress here
    resume_from_checkpoint=True          # Resume if interrupted
)

print("🎉 EVALUATION COMPLETED!")
print("Results saved in 'results_batched' variable")


print("💡 TO RUN:")
print("1. Uncomment the code above")
print("2. Run this cell")
print("3. Watch progress in real-time")
print("4. If interrupted, just run again to resume!")

print("\n⏱️ ESTIMATED TIME:")
print("• ~2-3 hours for 200 users (with rate limiting)")
print("• But progress is saved every ~15-20 minutes")
print("• So you can stop/resume anytime!")


🚀 STARTING BATCHED EVALUATION
This will evaluate 200 users in batches of 25
Progress will be saved after each batch!

🚀 BATCHED EVALUATION: 200 users in batches of 25
📁 Checkpoint file: evaluation_200_users.json
API Tier: basic (RPM: 500, TPM: 200000)
Max workers - Bias: 5, Trials: 2, Users: 1
🔍 Computing bias analysis...
Using precalculated bias scores...
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/8 (25 users)
Evaluating 25 users in parallel with max_workers=1...


User evaluation:   0%|                                   | 0/25 [00:00<?, ?it/s]


Running 20 randomization trials...
Executing 20 trials in parallel with max_workers=2...
Rate limiting: 2 workers, 0.150s delay, batch size: 10



Trials (batch 1/2): 100%|███████████████████████| 10/10 [00:12<00:00,  1.27s/it]


Batch 1 complete, waiting 1.2s before next batch...


...

Batch 1 complete, waiting 1.2s before next batch...



als (batch 2/2):  30%|███████▏                | 3/10 [00:29<01:07,  9.69s/it]

Rate limit hit, retrying in 1.0s (attempt 1/3)



User evaluation: 100%|███████████████████████| 25/25 [1:09:22<00:00, 166.50s/it]

Completed 20 successful trials out of 20 attempted
Debiasing applied within each trial based on input position. Creating final ranking from aggregated debiased scores...
Final ranking created. Top 3 items:
  1. Rob Roy (1995) (agg: 0.770, debiased: 0.622, avg_weight: 0.833)
  2. Walking and Talking (1996) (agg: 0.628, debiased: 0.541, avg_weight: 0.878)
  3. Stripes (1981) (agg: 0.665, debiased: 0.517, avg_weight: 0.817)
✅ Batch 8 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.3500 ± 0.4770
  NDCG@1:      0.3500 ± 0.4770
  NDCG@5:      0.5346 ± 0.4042
  NDCG@10:     0.5771 ± 0.3607
  NDCG@20:     0.6188 ± 0.3013
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.3500±0.4770

Accuracy Compa

In [13]:
# 📈 CHECK PROGRESS DURING EXECUTION
# Run this cell periodically to monitor progress

print("📈 CHECKING CURRENT PROGRESS...")
print("=" * 40)

# Check current status
current_status = analyzer.get_checkpoint_status("evaluation_200_users.json")

if current_status.get('status') != 'No checkpoint found':
    completed = current_status.get('completed_users', 0)
    progress = current_status.get('progress_percent', 0)
    
    print(f"👥 Users completed: {completed}/200")
    print(f"📊 Progress: {progress:.1f}%")
    print(f"🔄 Batches done: {current_status.get('last_batch_completed', 0)}/{current_status.get('total_batches', 0)}")
    
    if progress < 100:
        remaining_users = 200 - completed
        print(f"⏳ Remaining users: {remaining_users}")
        print("💡 Evaluation is still running or can be resumed")
    else:
        print("🎉 Evaluation completed!")
else:
    print("❌ No evaluation in progress")
    print("💡 Start the batched evaluation in the cell above")

print("\n💾 Checkpoint file: evaluation_200_users.json")
print("🔄 You can interrupt and resume anytime!")


📈 CHECKING CURRENT PROGRESS...
❌ No evaluation in progress
💡 Start the batched evaluation in the cell above

💾 Checkpoint file: evaluation_200_users.json
🔄 You can interrupt and resume anytime!


In [ ]:
# 🎉 HANDLING COMPLETED RESULTS
# What to do when your batched evaluation finishes

print("🎉 HANDLING COMPLETED RESULTS")
print("=" * 40)

# Check if we have results (this will be populated after evaluation completes)
try:
    if 'results_batched' in locals():
        print("✅ Results available in 'results_batched' variable!")
        
        # Display key metrics
        eval_results = results_batched['our_method_evaluation']
        
        print(f"\n📊 FINAL RESULTS:")
        print(f"  Accuracy:  {eval_results['accuracy']['mean']:.4f} ± {eval_results['accuracy']['std']:.4f}")
        print(f"  NDCG@1:    {eval_results['ndcg_1']['mean']:.4f} ± {eval_results['ndcg_1']['std']:.4f}")
        print(f"  NDCG@5:    {eval_results['ndcg_5']['mean']:.4f} ± {eval_results['ndcg_5']['std']:.4f}")
        print(f"  NDCG@10:   {eval_results['ndcg_10']['mean']:.4f} ± {eval_results['ndcg_10']['std']:.4f}")
        print(f"  NDCG@20:   {eval_results['ndcg_20']['mean']:.4f} ± {eval_results['ndcg_20']['std']:.4f}")
        
        batch_info = results_batched['batch_info']
        print(f"\n📈 EVALUATION INFO:")
        print(f"  Users evaluated: {batch_info['completed_users']}")
        print(f"  Batches completed: {batch_info['last_batch_completed']}/{batch_info['total_batches']}")
        
        # Save results to file
        import json
        with open('final_results_200_users.json', 'w') as f:
            json.dump(results_batched, f, indent=2, default=str)
        print(f"\n💾 Results saved to: final_results_200_users.json")
        
    else:
        print("⏳ Results not yet available")
        print("Complete the batched evaluation first!")
        
except NameError:
    print("⏳ Evaluation not started yet")
    print("Run the batched evaluation cell first!")

print("\n🔧 WHAT YOU CAN DO WITH RESULTS:")
print("• Compare with STELLA and other benchmarks")
print("• Analyze different NDCG metrics") 
print("• Export to CSV/Excel for further analysis")
print("• Use in your research paper")

print("\n📊 EXAMPLE RESULT ACCESS:")
print("# Get accuracy")
print("accuracy = results_batched['our_method_evaluation']['accuracy']['mean']")
print()
print("# Get all NDCG scores")  
print("ndcg_scores = {")
print("    'ndcg_1': results_batched['our_method_evaluation']['ndcg_1']['mean'],")
print("    'ndcg_5': results_batched['our_method_evaluation']['ndcg_5']['mean'],")
print("    'ndcg_10': results_batched['our_method_evaluation']['ndcg_10']['mean'],")
print("    'ndcg_20': results_batched['our_method_evaluation']['ndcg_20']['mean']")
print("}")

print("\n🎊 CONGRATULATIONS!")
print("You've successfully completed a large-scale debiased ranking evaluation!")


In [ ]:
# 📁 CHECKPOINT FILE ANALYSIS
# Examining the saved checkpoint data for insights

print("📁 CHECKPOINT FILE ANALYSIS")
print("=" * 40)

print("✅ CHECKPOINT FILES ARE PRESERVED!")
print("• Files are NOT automatically deleted after completion")
print("• Rich data available for post-analysis")
print("• Comprehensive performance insights included")

print("\n🔍 ANALYZING YOUR CHECKPOINT FILE:")
print("Run this to get detailed insights from your evaluation:")
print()

# Analyze the checkpoint file if it exists
try:
    # Try to analyze the main evaluation checkpoint
    analysis = analyzer.analyze_checkpoint_file("evaluation_200_users.json")
    
    if analysis.get('status') == 'No checkpoint found':
        print("📂 No completed evaluation found yet.")
        print("💡 Run your batched evaluation first, then return here!")
        
        # Show what the analysis will include
        print("\n🔮 WHAT YOU'LL GET AFTER EVALUATION:")
        print("• 📊 Performance distribution (μ and σ for all metrics)")
        print("• 🏆 Best and worst performing users")
        print("• 📈 Accuracy distribution breakdown")
        print("• 🧠 Bias analysis summary")
        print("• 💾 File size and storage info")
        print("• 📋 Statistical insights")
    else:
        print("✅ Analysis complete! See results above.")
        
except Exception as e:
    print(f"⚠️ Could not analyze checkpoint: {e}")
    print("💡 Make sure you have completed an evaluation first!")

print("\n🔬 ADDITIONAL MANUAL ANALYSIS:")
print("# Load checkpoint data manually")
print("import json")
print("with open('evaluation_200_users.json', 'r') as f:")
print("    checkpoint_data = json.load(f)")
print()
print("# Extract specific insights")
print("all_results = checkpoint_data['all_user_results']")
print("accuracies = [r['accuracy'] for r in all_results]")
print("print(f'Mean accuracy: {np.mean(accuracies):.4f}')")

print("\n💡 RESEARCH VALUE:")
print("• 📈 Per-user performance analysis")
print("• 🎯 User behavior pattern identification")
print("• 📊 Performance variance analysis")
print("• 🔍 Outlier detection and analysis")
print("• 📋 Export data for publications")

print("\n🎯 TIP:")
print("These checkpoint files are research goldmines!")
print("Keep them for detailed post-analysis and paper writing.")


In [ ]:
# Example 1: Create different propensity score formulas

# Linear decay formula (higher positions get lower weights)
linear_propensity = analyzer.create_custom_propensity_scores(
    N=20,  # Number of positions
    formula="linear"
)
print("Linear Propensity Scores:", linear_propensity)

# Exponential decay formula
exponential_propensity = analyzer.create_custom_propensity_scores(
    N=20,
    formula="exponential"
)
print("Exponential Propensity Scores:", exponential_propensity)

# Custom formula (you can define your own function)
def custom_bias_function(position, total_positions):
    """Custom function that gives higher weights to middle positions"""
    mid_point = total_positions // 2
    distance_from_middle = abs(position - mid_point)
    return 1.0 / (1.0 + distance_from_middle * 0.1)

custom_propensity = analyzer.create_custom_propensity_scores(
    N=20,
    formula="custom",
    custom_function=custom_bias_function
)
print("Custom Propensity Scores:", custom_propensity)


In [ ]:
# Example 2: Reapply debiasing using saved checkpoint data

# Specify the checkpoint file from your previous run
checkpoint_file = "batched_evaluation_checkpoint_batch_25.json"

# Reapply debiasing with different propensity formulas
print("=" * 60)
print("🔄 REAPPLYING DEBIASING WITH DIFFERENT FORMULAS")
print("=" * 60)

# Test 1: Linear propensity scores
print("\n1️⃣ Testing Linear Propensity Scores:")
linear_results = analyzer.reapply_debiasing_from_checkpoint(
    checkpoint_file=checkpoint_file,
    new_propensity_scores=linear_propensity,
    aggregation_method="mean",
    save_results_to="linear_debiasing_results.json"
)

# Test 2: Exponential propensity scores
print("\n2️⃣ Testing Exponential Propensity Scores:")
exponential_results = analyzer.reapply_debiasing_from_checkpoint(
    checkpoint_file=checkpoint_file,
    new_propensity_scores=exponential_propensity,
    aggregation_method="mean",
    save_results_to="exponential_debiasing_results.json"
)

# Test 3: Custom propensity scores
print("\n3️⃣ Testing Custom Propensity Scores:")
custom_results = analyzer.reapply_debiasing_from_checkpoint(
    checkpoint_file=checkpoint_file,
    new_propensity_scores=custom_propensity,
    aggregation_method="mean",
    save_results_to="custom_debiasing_results.json"
)


In [ ]:
# Example 3: Compare results from different debiasing methods

import pandas as pd
import matplotlib.pyplot as plt

# Compile results for comparison
results_comparison = []

# Add results from each method
methods = [
    ("Linear", linear_results),
    ("Exponential", exponential_results),
    ("Custom", custom_results)
]

for method_name, results in methods:
    if 'recomputed_evaluation' in results:
        eval_data = results['recomputed_evaluation']
        results_comparison.append({
            'Method': method_name,
            'Accuracy': eval_data['accuracy']['mean'],
            'Accuracy_Std': eval_data['accuracy']['std'],
            'NDCG@1': eval_data['ndcg_1']['mean'],
            'NDCG@1_Std': eval_data['ndcg_1']['std'],
            'NDCG@5': eval_data['ndcg_5']['mean'],
            'NDCG@5_Std': eval_data['ndcg_5']['std'],
            'NDCG@10': eval_data['ndcg_10']['mean'],
            'NDCG@10_Std': eval_data['ndcg_10']['std'],
            'NDCG@20': eval_data['ndcg_20']['mean'],
            'NDCG@20_Std': eval_data['ndcg_20']['std']
        })

# Create comparison DataFrame
comparison_df = pd.DataFrame(results_comparison)
print("📊 DEBIASING METHODS COMPARISON:")
print(comparison_df.round(4))

# Plot comparison
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
metrics = ['Accuracy', 'NDCG@1', 'NDCG@5', 'NDCG@10', 'NDCG@20']

for i, metric in enumerate(metrics):
    ax = axes[i]
    ax.bar(comparison_df['Method'], comparison_df[metric], 
           yerr=comparison_df[f'{metric}_Std'], capsize=5)
    ax.set_title(f'{metric} Comparison')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print detailed analysis
print("\n🔍 DETAILED ANALYSIS:")
for i, row in comparison_df.iterrows():
    print(f"\n{row['Method']} Method:")
    print(f"  Accuracy: {row['Accuracy']:.4f} ± {row['Accuracy_Std']:.4f}")
    print(f"  NDCG@1:   {row['NDCG@1']:.4f} ± {row['NDCG@1_Std']:.4f}")
    print(f"  NDCG@5:   {row['NDCG@5']:.4f} ± {row['NDCG@5_Std']:.4f}")
    print(f"  NDCG@10:  {row['NDCG@10']:.4f} ± {row['NDCG@10_Std']:.4f}")
    print(f"  NDCG@20:  {row['NDCG@20']:.4f} ± {row['NDCG@20_Std']:.4f}")
